In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from collections import defaultdict, deque
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor

In [2]:
# =========================
# PATH & LOAD DATA
# =========================
BASE_PATH = r"C:\Users\Gusti Jogish\Downloads\Gammafest\dataset"

TRAIN_PATH = fr"{BASE_PATH}\train.csv"
TEST_PATH = fr"{BASE_PATH}\test.csv"
SAMPLE_SUB_PATH = fr"{BASE_PATH}\sample submission.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Sample shape:", sample_sub.shape)

Train shape: (78772, 47)
Test shape : (42422, 20)
Sample shape: (42422, 3)


## Data Understanding

In [3]:
missing_train = pd.DataFrame({
    "train_missing_count": train.isna().sum(),
    "train_missing_pct": (train.isna().sum() / len(train)) * 100
})

missing_test = pd.DataFrame({
    "test_missing_count": test.isna().sum(),
    "test_missing_pct": (test.isna().sum() / len(test)) * 100
})

missing_summary = missing_train.join(missing_test, how="outer").fillna(0)
missing_summary = missing_summary.sort_values(["test_missing_pct", "train_missing_pct"], ascending=False)

display(missing_summary.head(20))

,train_missing_count,train_missing_pct,test_missing_count,test_missing_pct
distance_travel_opp,30392,38.582237,16975.0,40.014615
distance_travel_team,30392,38.582237,16975.0,40.014615
gdp_per_capita_opp,25957,32.952064,14664.0,34.566970
gdp_per_capita_team,25957,32.952064,14664.0,34.566970
altitude_venue,20554,26.093028,10906.0,25.708359
temperature_venue,17074,21.675215,5594.0,13.186554
population_opp,14872,18.879805,3372.0,7.948706
population_team,14872,18.879805,3372.0,7.948706
rank_diff,43902,55.733002,0.0,0.000000
rank_opponent,41946,53.249886,0.0,0.000000


## AW-MAE Metric

In [4]:
TOURNAMENT_WEIGHTS = {
    "FIFA World Cup": 2.00,
    "AFC Asian Cup": 1.80,
    "African Cup of Nations": 1.80,
    "UEFA Euro": 1.90,
    "Copa América": 1.90,
    "Confederations Cup": 1.70,
    "Olympic Games": 1.50,
    "Friendly": 0.96
}
DEFAULT_WEIGHT = 1.20

def get_match_outcome(team_goals, opp_goals):
    if team_goals > opp_goals:
        return 1
    elif team_goals < opp_goals:
        return -1
    return 0

def compute_match_loss(y_true_team, y_true_opp, y_pred_team, y_pred_opp, tournament_name):
    y_pred_team = max(0, int(y_pred_team))
    y_pred_opp = max(0, int(y_pred_opp))

    mae = (abs(y_true_team - y_pred_team) + abs(y_true_opp - y_pred_opp)) / 2

    exact = int((y_true_team == y_pred_team) and (y_true_opp == y_pred_opp))
    outcome = int(get_match_outcome(y_true_team, y_true_opp) == get_match_outcome(y_pred_team, y_pred_opp))
    gd = int((y_true_team - y_true_opp) == (y_pred_team - y_pred_opp))

    penalty = 0.30 * (1 - exact) + 0.25 * (1 - outcome) + 0.15 * (1 - gd)
    multiplier = 1.0 if outcome == 1 else 1.5
    raw_loss = mae + penalty
    loss = (raw_loss * multiplier) ** 1.5

    weight = TOURNAMENT_WEIGHTS.get(tournament_name, DEFAULT_WEIGHT)
    return loss, weight

def aw_mae_score(df_eval, true_team_col, true_opp_col, pred_team_col, pred_opp_col, tournament_col="tournament"):
    weighted_losses = []
    weights = []

    for _, row in df_eval.iterrows():
        loss, weight = compute_match_loss(
            row[true_team_col],
            row[true_opp_col],
            row[pred_team_col],
            row[pred_opp_col],
            row[tournament_col]
        )
        weighted_losses.append(loss * weight)
        weights.append(weight)

    return np.sum(weighted_losses) / np.sum(weights)

## Preprocessing

In [5]:
train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])

for df in [train, test]:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
    df["decade"] = (df["year"] // 10) * 10

print(train[["date", "year", "month", "dayofweek", "decade"]].head())

        date  year  month  dayofweek  decade
0 1872-11-30  1872     11          5    1870
1 1872-11-30  1872     11          5    1870
2 1873-03-08  1873      3          5    1870
3 1873-03-08  1873      3          5    1870
4 1874-03-07  1874      3          5    1870


In [6]:
MODERN_YEAR = 1990
train_modern = train[train["year"] >= MODERN_YEAR].copy().reset_index(drop=True)

print("Original train shape :", train.shape)
print("Modern train shape   :", train_modern.shape)
print("Year min-max modern  :", train_modern["year"].min(), "-", train_modern["year"].max())

Original train shape : (78772, 53)
Modern train shape   : (43610, 53)
Year min-max modern  : 1990 - 2011


## Historical Features

In [7]:
def safe_avg(values, last_n=None):
    if values is None:
        return np.nan
    arr = list(values)
    if last_n is not None:
        arr = arr[-last_n:]
    if len(arr) == 0:
        return np.nan
    return float(np.mean(arr))

def build_historical_features(train_df, test_df):
    train_tmp = train_df.copy()
    test_tmp = test_df.copy()

    train_tmp["_is_train"] = 1
    test_tmp["_is_train"] = 0

    for col in ["team_goals", "opp_goals"]:
        if col not in test_tmp.columns:
            test_tmp[col] = np.nan

    combined = pd.concat([train_tmp, test_tmp], ignore_index=True, sort=False)
    combined["_orig_order"] = np.arange(len(combined))
    sort_cols = [c for c in ["date", "match_id", "team", "opponent", "_is_train"] if c in combined.columns]
    combined = combined.sort_values(sort_cols).reset_index(drop=True)

    team_hist = defaultdict(lambda: {
        "gf": deque(maxlen=10),
        "ga": deque(maxlen=10),
        "pts": deque(maxlen=10),
        "gd": deque(maxlen=10),
        "win": deque(maxlen=10),
        "last_date": None,
        "matches": 0
    })

    pair_hist = defaultdict(lambda: {
        "pts": deque(maxlen=5),
        "gd": deque(maxlen=5),
        "gf": deque(maxlen=5),
        "matches": 0
    })

    feat_rows = []

    for _, row in combined.iterrows():
        cur_date = row["date"]
        team = row["team"]
        opp = row["opponent"]

        th = team_hist[team]
        oh = team_hist[opp]
        ph = pair_hist[(team, opp)]

        team_days = np.nan if th["last_date"] is None else (cur_date - th["last_date"]).days
        opp_days = np.nan if oh["last_date"] is None else (cur_date - oh["last_date"]).days

        feat_rows.append({
            "team_hist_matches": th["matches"],
            "opp_hist_matches": oh["matches"],

            "team_hist_gf_avg5": safe_avg(th["gf"], 5),
            "team_hist_ga_avg5": safe_avg(th["ga"], 5),
            "team_hist_pts_avg5": safe_avg(th["pts"], 5),
            "team_hist_gd_avg5": safe_avg(th["gd"], 5),
            "team_hist_win_rate5": safe_avg(th["win"], 5),

            "opp_hist_gf_avg5": safe_avg(oh["gf"], 5),
            "opp_hist_ga_avg5": safe_avg(oh["ga"], 5),
            "opp_hist_pts_avg5": safe_avg(oh["pts"], 5),
            "opp_hist_gd_avg5": safe_avg(oh["gd"], 5),
            "opp_hist_win_rate5": safe_avg(oh["win"], 5),

            "team_hist_gf_avg10": safe_avg(th["gf"], 10),
            "team_hist_ga_avg10": safe_avg(th["ga"], 10),
            "opp_hist_gf_avg10": safe_avg(oh["gf"], 10),
            "opp_hist_ga_avg10": safe_avg(oh["ga"], 10),

            "team_days_since_last_match": team_days,
            "opp_days_since_last_match": opp_days,

            "h2h_matches": ph["matches"],
            "h2h_pts_avg5": safe_avg(ph["pts"], 5),
            "h2h_gd_avg5": safe_avg(ph["gd"], 5),
            "h2h_gf_avg5": safe_avg(ph["gf"], 5)
        })

        if row["_is_train"] == 1 and pd.notna(row["team_goals"]) and pd.notna(row["opp_goals"]):
            gf = int(row["team_goals"])
            ga = int(row["opp_goals"])
            pts = 3 if gf > ga else (1 if gf == ga else 0)
            gd = gf - ga
            win = 1 if gf > ga else 0

            th["gf"].append(gf)
            th["ga"].append(ga)
            th["pts"].append(pts)
            th["gd"].append(gd)
            th["win"].append(win)
            th["last_date"] = cur_date
            th["matches"] += 1

            ph["pts"].append(pts)
            ph["gd"].append(gd)
            ph["gf"].append(gf)
            ph["matches"] += 1

    feat_df = pd.DataFrame(feat_rows)
    combined = pd.concat([combined.reset_index(drop=True), feat_df.reset_index(drop=True)], axis=1)
    combined["form_strength_diff5"] = combined["team_hist_gd_avg5"] - combined["opp_hist_gd_avg5"]
    combined["attack_strength_diff5"] = combined["team_hist_gf_avg5"] - combined["opp_hist_gf_avg5"]
    combined["defense_weakness_diff5"] = combined["team_hist_ga_avg5"] - combined["opp_hist_ga_avg5"]
    combined["win_rate_diff5"] = combined["team_hist_win_rate5"] - combined["opp_hist_win_rate5"]
    combined["rest_days_diff"] = combined["team_days_since_last_match"] - combined["opp_days_since_last_match"]

    train_out = combined[combined["_is_train"] == 1].copy()
    test_out = combined[combined["_is_train"] == 0].copy()

    helper_cols = ["_is_train", "_orig_order"]
    train_out = train_out.drop(columns=[c for c in helper_cols if c in train_out.columns]).reset_index(drop=True)
    test_out = test_out.drop(columns=[c for c in helper_cols if c in test_out.columns]).reset_index(drop=True)

    return train_out, test_out

train_feat, test_feat = build_historical_features(train_modern, test)

print("Train feat shape:", train_feat.shape)
print("Test feat shape :", test_feat.shape)

hist_cols = [c for c in train_feat.columns if c.startswith("team_hist_") or c.startswith("opp_hist_") or c.startswith("h2h_") or c.endswith("_diff5") or c == "rest_days_diff"]
print("Jumlah historical features:", len(hist_cols))
print(hist_cols)

Train feat shape: (43610, 80)
Test feat shape : (42422, 80)
Jumlah historical features: 27
['h2h_points_last5', 'h2h_gd_last5', 'team_hist_matches', 'opp_hist_matches', 'team_hist_gf_avg5', 'team_hist_ga_avg5', 'team_hist_pts_avg5', 'team_hist_gd_avg5', 'team_hist_win_rate5', 'opp_hist_gf_avg5', 'opp_hist_ga_avg5', 'opp_hist_pts_avg5', 'opp_hist_gd_avg5', 'opp_hist_win_rate5', 'team_hist_gf_avg10', 'team_hist_ga_avg10', 'opp_hist_gf_avg10', 'opp_hist_ga_avg10', 'h2h_matches', 'h2h_pts_avg5', 'h2h_gd_avg5', 'h2h_gf_avg5', 'form_strength_diff5', 'attack_strength_diff5', 'defense_weakness_diff5', 'win_rate_diff5', 'rest_days_diff']


## Common + Engineered Features

In [8]:
core_features = [
    "opponent",
    "team",
    "gender",
    "population_opp",
    "population_team",
    "is_home",
    "tournament",
    "confederation_team",
    "confederation_opp",
    "gdp_per_capita_opp",
    "gdp_per_capita_team",
    "venue_country",
    "distance_travel_team",
    "distance_travel_opp",
    "altitude_venue",
    "temperature_venue",
    "day",
    "month",
    "year"
]

candidate_features = core_features + hist_cols

final_features = [col for col in candidate_features if col in train_feat.columns and col in test_feat.columns]
final_features = list(dict.fromkeys(final_features))

categorical_features = [col for col in final_features if train_feat[col].dtype == "object"]
numerical_features = [col for col in final_features if col not in categorical_features]

print("Jumlah final features:", len(final_features))
print(final_features)

print("\nCategorical:")
print(categorical_features)

print("\nNumerical (sample 20):")
print(numerical_features[:20])

Jumlah final features: 46
['opponent', 'team', 'gender', 'population_opp', 'population_team', 'is_home', 'tournament', 'confederation_team', 'confederation_opp', 'gdp_per_capita_opp', 'gdp_per_capita_team', 'venue_country', 'distance_travel_team', 'distance_travel_opp', 'altitude_venue', 'temperature_venue', 'day', 'month', 'year', 'h2h_points_last5', 'h2h_gd_last5', 'team_hist_matches', 'opp_hist_matches', 'team_hist_gf_avg5', 'team_hist_ga_avg5', 'team_hist_pts_avg5', 'team_hist_gd_avg5', 'team_hist_win_rate5', 'opp_hist_gf_avg5', 'opp_hist_ga_avg5', 'opp_hist_pts_avg5', 'opp_hist_gd_avg5', 'opp_hist_win_rate5', 'team_hist_gf_avg10', 'team_hist_ga_avg10', 'opp_hist_gf_avg10', 'opp_hist_ga_avg10', 'h2h_matches', 'h2h_pts_avg5', 'h2h_gd_avg5', 'h2h_gf_avg5', 'form_strength_diff5', 'attack_strength_diff5', 'defense_weakness_diff5', 'win_rate_diff5', 'rest_days_diff']

Categorical:
['opponent', 'team', 'gender', 'tournament', 'confederation_team', 'confederation_opp', 'venue_country']

N

## Imputation

In [9]:
train_imp = train_feat.copy()
test_imp = test_feat.copy()

for col in categorical_features:
    train_imp[col] = train_imp[col].fillna("Unknown")
    test_imp[col] = test_imp[col].fillna("Unknown")

for col in numerical_features:
    med = train_imp[col].median()
    train_imp[col] = train_imp[col].fillna(med)
    test_imp[col] = test_imp[col].fillna(med)

print("Train missing remaining:", train_imp[final_features].isna().sum().sum())
print("Test missing remaining :", test_imp[final_features].isna().sum().sum())

Train missing remaining: 0
Test missing remaining : 0


## Split

In [10]:
train_imp = train_imp.sort_values("date").reset_index(drop=True)

split_date = train_imp["date"].quantile(0.85)

train_part = train_imp[train_imp["date"] < split_date].copy()
valid_part = train_imp[train_imp["date"] >= split_date].copy()

X_train = train_part[final_features].copy()
X_valid = valid_part[final_features].copy()

y_train_team = train_part["team_goals"].copy()
y_train_opp = train_part["opp_goals"].copy()

y_valid_team = valid_part["team_goals"].copy()
y_valid_opp = valid_part["opp_goals"].copy()

print("Split date:", split_date)
print("Train shape:", X_train.shape)
print("Valid shape:", X_valid.shape)

Split date: 2008-10-20 00:00:00
Train shape: (37066, 46)
Valid shape: (6544, 46)


## Train `team_goals`

In [11]:
team_model = CatBoostRegressor(
    iterations=900,
    learning_rate=0.04,
    depth=7,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=42,
    verbose=100
)

team_model.fit(
    X_train,
    y_train_team,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid_team),
    use_best_model=True
)

0:	learn: 1.1547318	test: 1.1076642	best: 1.1076642 (0)	total: 108ms	remaining: 1m 36s
100:	learn: 0.9780199	test: 0.9638445	best: 0.9638445 (100)	total: 4.38s	remaining: 34.6s
200:	learn: 0.9518895	test: 0.9558593	best: 0.9558593 (200)	total: 8.56s	remaining: 29.8s
300:	learn: 0.9350118	test: 0.9529643	best: 0.9529643 (300)	total: 12.7s	remaining: 25.3s
400:	learn: 0.9169817	test: 0.9511764	best: 0.9511764 (400)	total: 16.9s	remaining: 21s
500:	learn: 0.9028286	test: 0.9504528	best: 0.9503502 (494)	total: 21.2s	remaining: 16.9s
600:	learn: 0.8914070	test: 0.9498750	best: 0.9498750 (600)	total: 25.4s	remaining: 12.6s
700:	learn: 0.8818037	test: 0.9492772	best: 0.9491777 (692)	total: 29.5s	remaining: 8.39s
800:	learn: 0.8726570	test: 0.9488281	best: 0.9484070 (735)	total: 33.6s	remaining: 4.15s
899:	learn: 0.8649708	test: 0.9489250	best: 0.9484070 (735)	total: 38.1s	remaining: 0us

bestTest = 0.9484070099
bestIteration = 735

Shrink model to first 736 iterations.


## Train `opp_goals`

In [12]:
opp_model = CatBoostRegressor(
    iterations=900,
    learning_rate=0.04,
    depth=7,
    loss_function="MAE",
    eval_metric="MAE",
    random_seed=42,
    verbose=100
)

opp_model.fit(
    X_train,
    y_train_opp,
    cat_features=categorical_features,
    eval_set=(X_valid, y_valid_opp),
    use_best_model=True
)

0:	learn: 1.1548354	test: 1.1081471	best: 1.1081471 (0)	total: 51.5ms	remaining: 46.3s
100:	learn: 0.9988199	test: 0.9845435	best: 0.9845435 (100)	total: 4.28s	remaining: 33.9s
200:	learn: 0.9696232	test: 0.9726585	best: 0.9726585 (200)	total: 8.6s	remaining: 29.9s
300:	learn: 0.9496400	test: 0.9671483	best: 0.9671483 (300)	total: 13s	remaining: 25.8s
400:	learn: 0.9324956	test: 0.9639575	best: 0.9639496 (398)	total: 17.4s	remaining: 21.6s
500:	learn: 0.9177700	test: 0.9626926	best: 0.9626926 (500)	total: 21.7s	remaining: 17.3s
600:	learn: 0.9060411	test: 0.9619683	best: 0.9618900 (580)	total: 26s	remaining: 12.9s
700:	learn: 0.8957630	test: 0.9616172	best: 0.9614770 (695)	total: 30s	remaining: 8.52s
800:	learn: 0.8864559	test: 0.9614109	best: 0.9610554 (778)	total: 34.1s	remaining: 4.21s
899:	learn: 0.8780494	test: 0.9611049	best: 0.9609891 (889)	total: 38.1s	remaining: 0us

bestTest = 0.96098915
bestIteration = 889

Shrink model to first 890 iterations.


## Validation + Post-processing Tuning

In [13]:
valid_pred_team_raw = team_model.predict(X_valid)
valid_pred_opp_raw = opp_model.predict(X_valid)

print("Raw prediction stats:")
print("team -> min:", np.min(valid_pred_team_raw), "max:", np.max(valid_pred_team_raw), "mean:", np.mean(valid_pred_team_raw))
print("opp  -> min:", np.min(valid_pred_opp_raw), "max:", np.max(valid_pred_opp_raw), "mean:", np.mean(valid_pred_opp_raw))

def apply_postprocess(pred_team, pred_opp, cap=4, draw_margin=0.18, low_draw_threshold=0.85, high_draw_threshold=1.75):
    out_team = []
    out_opp = []

    for t, o in zip(pred_team, pred_opp):
        t = float(np.clip(t, 0, cap))
        o = float(np.clip(o, 0, cap))
        diff = t - o
        avg_goals = (t + o) / 2

        if abs(diff) <= draw_margin:
            if avg_goals <= low_draw_threshold:
                pt, po = 0, 0
            elif avg_goals <= high_draw_threshold:
                pt, po = 1, 1
            else:
                pt, po = 2, 2
        else:
            pt = int(np.round(t))
            po = int(np.round(o))

            if pt == po:
                if diff > 0:
                    pt = min(cap, pt + 1)
                else:
                    po = min(cap, po + 1)

        out_team.append(int(max(0, pt)))
        out_opp.append(int(max(0, po)))

    return np.array(out_team), np.array(out_opp)

grid_results = []

for cap in [4, 5]:
    for draw_margin in [0.10, 0.15, 0.20, 0.25]:
        for low_draw_threshold in [0.65, 0.85, 1.00]:
            for high_draw_threshold in [1.50, 1.75, 2.00]:
                pred_t, pred_o = apply_postprocess(
                    valid_pred_team_raw,
                    valid_pred_opp_raw,
                    cap=cap,
                    draw_margin=draw_margin,
                    low_draw_threshold=low_draw_threshold,
                    high_draw_threshold=high_draw_threshold
                )

                tmp = valid_part.copy()
                tmp["pred_team_goals"] = pred_t
                tmp["pred_opp_goals"] = pred_o

                awmae = aw_mae_score(
                    tmp,
                    true_team_col="team_goals",
                    true_opp_col="opp_goals",
                    pred_team_col="pred_team_goals",
                    pred_opp_col="pred_opp_goals",
                    tournament_col="tournament"
                )

                overall_mae = (
                    (tmp["team_goals"] - tmp["pred_team_goals"]).abs() +
                    (tmp["opp_goals"] - tmp["pred_opp_goals"]).abs()
                ).mean() / 2

                grid_results.append({
                    "cap": cap,
                    "draw_margin": draw_margin,
                    "low_draw_threshold": low_draw_threshold,
                    "high_draw_threshold": high_draw_threshold,
                    "awmae": awmae,
                    "overall_mae": overall_mae
                })

grid_df = pd.DataFrame(grid_results).sort_values(["awmae", "overall_mae"], ascending=[True, True]).reset_index(drop=True)
display(grid_df.head(10))

best_params = grid_df.iloc[0].to_dict()
print("Best params:", best_params)

Raw prediction stats:
team -> min: -0.4034845206640234 max: 11.668833784143049 mean: 1.2824937175970719
opp  -> min: -0.6609319820938715 max: 8.508742410672792 mean: 1.2656662413514217


,cap,draw_margin,low_draw_threshold,high_draw_threshold,awmae,overall_mae
0,5,0.25,0.65,1.50,2.790868,0.949267
1,5,0.25,0.65,1.75,2.791153,0.949114
2,5,0.25,0.65,2.00,2.791508,0.949114
3,5,0.20,0.65,1.75,2.794770,0.953240
4,5,0.20,0.65,2.00,2.795125,0.953240
5,5,0.20,0.65,1.50,2.795249,0.953545
6,5,0.15,0.65,1.50,2.800299,0.960727
7,5,0.15,0.65,1.75,2.800375,0.960575
8,5,0.15,0.65,2.00,2.800730,0.960575
9,5,0.10,0.65,1.50,2.802113,0.965465


Best params: {'cap': 5.0, 'draw_margin': 0.25, 'low_draw_threshold': 0.65, 'high_draw_threshold': 1.5, 'awmae': 2.7908678540085243, 'overall_mae': 0.9492665036674817}


In [14]:
valid_pred_team, valid_pred_opp = apply_postprocess(
    valid_pred_team_raw,
    valid_pred_opp_raw,
    cap=int(best_params["cap"]),
    draw_margin=float(best_params["draw_margin"]),
    low_draw_threshold=float(best_params["low_draw_threshold"]),
    high_draw_threshold=float(best_params["high_draw_threshold"])
)

valid_eval = valid_part.copy()
valid_eval["pred_team_goals"] = valid_pred_team
valid_eval["pred_opp_goals"] = valid_pred_opp

awmae = aw_mae_score(
    valid_eval,
    true_team_col="team_goals",
    true_opp_col="opp_goals",
    pred_team_col="pred_team_goals",
    pred_opp_col="pred_opp_goals",
    tournament_col="tournament"
)

mae_team = mean_absolute_error(valid_eval["team_goals"], valid_eval["pred_team_goals"])
mae_opp = mean_absolute_error(valid_eval["opp_goals"], valid_eval["pred_opp_goals"])
overall_mae = (
    (valid_eval["team_goals"] - valid_eval["pred_team_goals"]).abs() +
    (valid_eval["opp_goals"] - valid_eval["pred_opp_goals"]).abs()
).mean() / 2

print("Validation MAE team   :", mae_team)
print("Validation MAE opp    :", mae_opp)
print("Validation Overall MAE:", overall_mae)
print("Validation AW-MAE     :", awmae)

display(valid_eval[[
    "date", "team", "opponent", "tournament",
    "team_goals", "opp_goals",
    "pred_team_goals", "pred_opp_goals"
]].head(20))

Validation MAE team   : 0.9457518337408313
Validation MAE opp    : 0.9527811735941321
Validation Overall MAE: 0.9492665036674817
Validation AW-MAE     : 2.7908678540085243


,date,team,opponent,tournament,team_goals,opp_goals,pred_team_goals,pred_opp_goals
37066,2008-10-20,Vietnam,Australia,AFF Championship,0.0,1.0,0,2
37067,2008-10-20,Australia,Vietnam,AFF Championship,1.0,0.0,2,1
37068,2008-10-20,Thailand,Myanmar,AFF Championship,3.0,0.0,2,1
37069,2008-10-20,Malaysia,Afghanistan,Merdeka Tournament,6.0,0.0,3,0
37070,2008-10-20,Afghanistan,Malaysia,Merdeka Tournament,0.0,6.0,1,3
37071,2008-10-20,Myanmar,Thailand,AFF Championship,0.0,3.0,1,2
37072,2008-10-21,Brunei,Timor-Leste,AFF Championship qualification,4.0,1.0,3,2
37073,2008-10-21,Timor-Leste,Brunei,AFF Championship qualification,1.0,4.0,1,2
37074,2008-10-21,Laos,Philippines,AFF Championship qualification,2.0,1.0,2,2
37075,2008-10-21,Philippines,Laos,AFF Championship qualification,1.0,2.0,2,1


## Feature Importance

In [15]:
fi_team = pd.DataFrame({
    "feature": final_features,
    "importance_team": team_model.get_feature_importance()
}).sort_values("importance_team", ascending=False)

fi_opp = pd.DataFrame({
    "feature": final_features,
    "importance_opp": opp_model.get_feature_importance()
}).sort_values("importance_opp", ascending=False)

fi_merge = fi_team.merge(fi_opp, on="feature", how="outer").fillna(0)
fi_merge["importance_avg"] = (fi_merge["importance_team"] + fi_merge["importance_opp"]) / 2
fi_merge = fi_merge.sort_values("importance_avg", ascending=False)

display(fi_merge.head(30))

,feature,importance_team,importance_opp,importance_avg
32,team,7.930236,9.918836,8.924536
11,gender,6.374005,6.249746,6.311876
8,form_strength_diff5,3.819480,8.332718,6.076099
28,opponent,5.864392,4.878191,5.371292
5,defense_weakness_diff5,8.816489,1.540766,5.178627
18,is_home,5.027562,4.663697,4.845630
21,opp_hist_ga_avg5,8.083493,0.995817,4.539655
24,opp_hist_gf_avg5,1.058355,7.699704,4.379029
13,h2h_gd_last5,4.790008,3.707252,4.248630
12,h2h_gd_avg5,3.355530,4.566526,3.961028


## Train Full Model for Submission

In [16]:
X_full = train_imp[final_features].copy()
y_full_team = train_imp["team_goals"].copy()
y_full_opp = train_imp["opp_goals"].copy()
X_test = test_imp[final_features].copy()

best_iter_team = team_model.get_best_iteration()
best_iter_opp = opp_model.get_best_iteration()

final_team_model = CatBoostRegressor(
    iterations=best_iter_team if best_iter_team is not None and best_iter_team > 0 else 900,
    learning_rate=0.04,
    depth=7,
    loss_function="MAE",
    random_seed=42,
    verbose=100
)

final_opp_model = CatBoostRegressor(
    iterations=best_iter_opp if best_iter_opp is not None and best_iter_opp > 0 else 900,
    learning_rate=0.04,
    depth=7,
    loss_function="MAE",
    random_seed=42,
    verbose=100
)

final_team_model.fit(X_full, y_full_team, cat_features=categorical_features)
final_opp_model.fit(X_full, y_full_opp, cat_features=categorical_features)

0:	learn: 1.1470929	total: 51.4ms	remaining: 37.7s
100:	learn: 0.9757233	total: 4.28s	remaining: 26.9s
200:	learn: 0.9519631	total: 8.6s	remaining: 22.8s
300:	learn: 0.9353538	total: 13s	remaining: 18.7s
400:	learn: 0.9214594	total: 17.5s	remaining: 14.6s
500:	learn: 0.9086477	total: 21.9s	remaining: 10.2s
600:	learn: 0.8971710	total: 26.2s	remaining: 5.84s
700:	learn: 0.8880704	total: 30.4s	remaining: 1.48s
734:	learn: 0.8853648	total: 31.9s	remaining: 0us
0:	learn: 1.1482789	total: 41.7ms	remaining: 37s
100:	learn: 0.9933257	total: 4.21s	remaining: 32.9s
200:	learn: 0.9646065	total: 8.97s	remaining: 30.7s
300:	learn: 0.9475955	total: 13.5s	remaining: 26.4s
400:	learn: 0.9319435	total: 17.8s	remaining: 21.6s
500:	learn: 0.9191429	total: 22.1s	remaining: 17.1s
600:	learn: 0.9090023	total: 26.9s	remaining: 12.9s
700:	learn: 0.9005852	total: 31.1s	remaining: 8.34s
800:	learn: 0.8917427	total: 35.4s	remaining: 3.89s
888:	learn: 0.8849718	total: 39.1s	remaining: 0us


In [17]:
test_pred_team_raw = final_team_model.predict(X_test)
test_pred_opp_raw = final_opp_model.predict(X_test)

test_pred_team, test_pred_opp = apply_postprocess(
    test_pred_team_raw,
    test_pred_opp_raw,
    cap=int(best_params["cap"]),
    draw_margin=float(best_params["draw_margin"]),
    low_draw_threshold=float(best_params["low_draw_threshold"]),
    high_draw_threshold=float(best_params["high_draw_threshold"])
)

submission = sample_sub.copy()
submission["team_goals"] = test_pred_team.astype(int)
submission["opp_goals"] = test_pred_opp.astype(int)

submission.to_csv("submission_gammafest_hist_postprocess.csv", index=False)

print(submission.head())
print("\nSaved: submission_gammafest_hist_postprocess.csv")

                   Id  team_goals  opp_goals
0  M034984_Seychelles           1          2
1   M034984_Mauritius           2          1
2     M034985_Comoros           1          2
3    M034985_Maldives           1          1
4     M034986_Réunion           1          2

Saved: submission_gammafest_hist_postprocess.csv
